# Exploratory Data Analysis -- Classroom Engagement Telemetry

This notebook provides a visual overview of the annotated dataset used to train and evaluate the MobileNetV2 posture classifier.

In [ ]:
import random
from pathlib import Path
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PROJECT_ROOT = Path("..")
CROPS_DIR = PROJECT_ROOT / "data" / "crops"
SPLITS = ["train", "val", "test"]
CLASSES = ["0_oriented", "1_diverted", "2_obscured"]
SEED = 42

## 1. Class Distribution Across Splits

We count the number of annotated images per class in each split to understand the data balance.

In [ ]:
counts = defaultdict(dict)
for split in SPLITS:
    for cls in CLASSES:
        folder = CROPS_DIR / split / cls
        n = len(list(folder.glob("*.jpg"))) if folder.exists() else 0
        counts[split][cls] = n

x = np.arange(len(SPLITS))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
for i, cls in enumerate(CLASSES):
    vals = [counts[s][cls] for s in SPLITS]
    bars = ax.bar(x + i * width, vals, width, label=cls)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                str(v), ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x + width)
ax.set_xticklabels(SPLITS, fontsize=12)
ax.set_ylabel("Number of Samples", fontsize=12)
ax.set_title("Class Distribution Across Train / Val / Test Splits", fontsize=14, fontweight="bold")
ax.legend(title="Class")
plt.tight_layout()
plt.show()

### Observations

The training set is **heavily imbalanced**:
- `1_diverted` is the majority class.
- `2_obscured` is severely underrepresented (often < 20 samples).

This imbalance justifies our use of **Weighted `CrossEntropyLoss`** during training, where each class weight is computed as `total_samples / (num_classes * class_count)`. Without this, the model would trivially predict the majority class and still achieve a misleadingly high accuracy.

## 2. Sample Images From Each Training Class

A 3x3 grid showing three randomly selected crops per class from the **training** split. This helps verify annotation quality and understand visual differences between classes.

In [ ]:
random.seed(SEED)

fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for row, cls in enumerate(CLASSES):
    folder = CROPS_DIR / "train" / cls
    images = sorted(folder.glob("*.jpg"))
    samples = random.sample(images, min(3, len(images)))

    for col in range(3):
        ax = axes[row][col]
        if col < len(samples):
            img = Image.open(samples[col])
            ax.imshow(img)
            ax.set_title(samples[col].name, fontsize=7)
        ax.axis("off")

    axes[row][0].set_ylabel(cls, fontsize=12, rotation=0, labelpad=80, va="center")

fig.suptitle("Training Set -- Sample Images per Class", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 3. Dataset Limitations

Key constraints to acknowledge:

1. **Small dataset size.** The entire annotated corpus is approximately 300 images across all splits. This is a proof-of-concept scale; production engagement systems would require thousands of samples with inter-annotator agreement.

2. **Single annotator.** All labels were assigned by one person, introducing potential subjective bias in the boundary between *oriented* and *diverted* postures.

3. **Limited camera viewpoints.** Training data comes from only two videos (fixed camera angles). The held-out test video may exhibit domain shift in lighting, seating density, or camera perspective, which directly impacts generalization.

4. **Class imbalance.** The `2_obscured` class is thin enough that per-class metrics (especially Recall) should be interpreted with caution due to small-sample variance.